# Data Preparation

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from config import (
  UNDERSTANDING_PROVINCES_CSV,
  UNDERSTANDING_REGENCIES_CSV,
  FEATURE_EVALUATION_JSON,
  FEATURE_SELECTION_JSON,
  PREPARED_PROVINCES_CSV,
  PREPARED_REGENCIES_CSV,
  SCALED_FEATURES_CSV
)

In [ ]:
MAX_VIF = 10.0
MIN_CV = 10.0
SKEWNESS_THRESHOLD = 2.0

In [ ]:
print(f"Parameter MAX_VIF            : {MAX_VIF}")
print(f"Parameter MIN_CV             : {MIN_CV}%")
print(f"Parameter SKEWNESS_THRESHOLD : {SKEWNESS_THRESHOLD}")

In [ ]:
df_prov = pd.read_csv(UNDERSTANDING_PROVINCES_CSV)
df_reg = pd.read_csv(UNDERSTANDING_REGENCIES_CSV)

## Pembersihan Data & Imputasi

### Standarisasi Teks

In [ ]:
df_prov['province_name'] = df_prov['province_name'].astype(str).str.strip().str.upper()
df_reg['regency_name'] = df_reg['regency_name'].astype(str).str.strip().str.upper()

### Imputasi Nilai Hilang (kNN Imputer)

In [ ]:
num_cols = df_reg.select_dtypes('number').columns
imputer = KNNImputer(n_neighbors=5)
df_reg[num_cols] = imputer.fit_transform(df_reg[num_cols])

os.makedirs(os.path.dirname(PREPARED_PROVINCES_CSV), exist_ok=True)
df_prov.to_csv(PREPARED_PROVINCES_CSV, index=False)
df_reg.to_csv(PREPARED_REGENCIES_CSV, index=False)

print(f"Total Baris Provinsi Prepared        : {len(df_prov)}")
print(f"Total Baris Kabupaten/Kota Prepared : {len(df_reg)}")

## Pemilihan Fitur

In [ ]:
with open(FEATURE_EVALUATION_JSON, 'r', encoding='utf-8') as f:
  feature_config = json.load(f)

vif_dict = feature_config.get('vif', {})
cv_dict = feature_config.get('variability_cv', {})
skew_dict = feature_config.get('skewness', {})

selected_features = [
  feat for feat, vif_val in vif_dict.items()
  if vif_val <= MAX_VIF and cv_dict.get(feat, 100.0) >= MIN_CV
]
eliminated_features = [feat for feat in vif_dict if feat not in selected_features]

log_transform_features = [
  feat for feat in selected_features
  if skew_dict.get(feat, 0) >= SKEWNESS_THRESHOLD
]

print(f"Fitur Terpilih (VIF <= {MAX_VIF}, CV >= {MIN_CV}%) : {selected_features}")
print(f"Fitur Dieliminasi                           : {eliminated_features}")
print(f"Fitur Transformasi Log (Skew >= {SKEWNESS_THRESHOLD})   : {log_transform_features}")

## Transformasi & Standardisasi Fitur

### Transformasi Logaritmik (Log1p)

In [ ]:
features_present = [col for col in selected_features if col in df_reg.columns]
X_df = df_reg[features_present].copy()

for col in log_transform_features:
  if col in X_df.columns:
    X_df[col] = np.log1p(np.maximum(X_df[col].values, 0))

print(X_df.describe().round(2).T.to_markdown())

### Standardisasi Fitur (StandardScaler)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values)

scaled_cols = [f"scaled_{c}" for c in features_present]
scaled_df = pd.DataFrame(X_scaled, columns=scaled_cols)
os.makedirs(os.path.dirname(SCALED_FEATURES_CSV), exist_ok=True)
scaled_df.to_csv(SCALED_FEATURES_CSV, index=False)

# Menyimpan hasil pemilihan fitur ke dalam file JSON
feature_selection = {
  "selected_features": selected_features,
  "eliminated_features": eliminated_features,
  "log_transform_features": log_transform_features,
  "scaled_feature_columns": scaled_cols
}

os.makedirs(os.path.dirname(FEATURE_SELECTION_JSON), exist_ok=True)
with open(FEATURE_SELECTION_JSON, 'w', encoding='utf-8') as f:
  json.dump(feature_selection, f, indent=2)

print(f"Total Fitur Terpilih Terstandarisasi : {len(features_present)}")
print(f"Bentuk Matriks Fitur                : {scaled_df.shape}")
print(f"Hasil feature selection disimpan di  : {FEATURE_SELECTION_JSON}")
print(scaled_df.head().to_markdown(index=False))